#### Preparation

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torchvision.models import convnext_small, ConvNeXt_Small_Weights
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import os

In [2]:
WEIGHTS_DIR = "../weights"

In [3]:
weights = ConvNeXt_Small_Weights.DEFAULT

mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mean,
        std=std
    )
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mean,
        std=std
    )
])

In [4]:
train_dataset = datasets.ImageFolder(
    r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\val",
    transform=val_transform
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

class_names = train_dataset.classes
num_classes = len(class_names)

print(f"Train classes: {class_names}")
print(f"Validation classes: {val_dataset.classes}")
print(f"Number of classes: {num_classes}")



Train classes: ['Healthy', 'Spider Mites', 'leaf blight', 'leaf spot']
Validation classes: ['Healthy', 'Spider Mites', 'leaf blight', 'leaf spot']
Number of classes: 4


In [5]:
model = convnext_small(weights=weights)


In [6]:
in_features = model.classifier[2].in_features

model.classifier[2] = nn.Linear(in_features, num_classes)


#### Training 1 (Freeze-Backbone)

In [7]:
for param in model.features.parameters():
    param.requires_grad = False


In [8]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model.to(DEVICE)


ConvNeXt(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      (1): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True)
    )
    (1): Sequential(
      (0): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=96, out_features=384, bias=True)
          (4): GELU(approximate='none')
          (5): Linear(in_features=384, out_features=96, bias=True)
          (6): Permute()
        )
        (stochastic_depth): StochasticDepth(p=0.0, mode=row)
      )
      (1): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.classifier.parameters(),
    lr=1e-4
)


In [10]:
def train_one_epoch(loader, epoch, epochs):
    model.train()
    
    train_loss = 0
    train_preds = []
    train_labels = []

    for images, labels in tqdm(loader, desc=f"Training Epoch {epoch+1}/{epochs}"):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        preds = outputs.argmax(dim=1)

        train_preds.extend(preds.cpu().numpy())
        train_labels.extend(labels.cpu().numpy())

        train_acc = accuracy_score(train_labels, train_preds)


    return train_loss / len(loader), train_acc


In [11]:
def validate(loader):
    model.eval()
    
    val_loss = 0
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating"):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            preds = outputs.argmax(dim=1)

            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
        
        val_acc = accuracy_score(val_labels, val_preds)

        precision, recall, f1, _ = precision_recall_fscore_support(
        val_labels, val_preds, average='weighted'
        )   

    return precision, recall, f1, val_loss / len(loader), val_acc

In [12]:
EPOCHS_STAGE_FREEZE = 8

print("\n-----------Starting Phase 1 Training-----------\n")


for epoch in range(EPOCHS_STAGE_FREEZE):

    train_loss, train_acc = train_one_epoch(train_loader, epoch, EPOCHS_STAGE_FREEZE)
    precision, recall, f1, val_loss, val_acc = validate(val_loader)

    print(f"\nEpoch [{epoch+1}/{EPOCHS_STAGE_FREEZE}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}\n\n")



-----------Starting Phase 1 Training-----------



Validating: 100%|██████████| 8/8 [00:50<00:00,  6.33s/it]



Epoch [1/8]
Train Loss: 0.9347 | Train Acc: 0.6458
Val Loss: 0.6327 | Val Acc: 0.7833
Precision: 0.8200 | Recall: 0.7833 | F1: 0.7762




Validating: 100%|██████████| 8/8 [00:47<00:00,  5.96s/it]



Epoch [2/8]
Train Loss: 0.5386 | Train Acc: 0.8365
Val Loss: 0.4607 | Val Acc: 0.8458
Precision: 0.8587 | Recall: 0.8458 | F1: 0.8434




...


Epoch [6/8]
Train Loss: 0.2903 | Train Acc: 0.9115
Val Loss: 0.2967 | Val Acc: 0.9000
Precision: 0.9055 | Recall: 0.9000 | F1: 0.8994




Validating: 100%|██████████| 8/8 [00:47<00:00,  5.92s/it]



Epoch [7/8]
Train Loss: 0.2714 | Train Acc: 0.9313
Val Loss: 0.2775 | Val Acc: 0.9083
Precision: 0.9130 | Recall: 0.9083 | F1: 0.9075




Validating: 100%|██████████| 8/8 [00:50<00:00,  6.27s/it]


Epoch [8/8]
Train Loss: 0.2505 | Train Acc: 0.9344
Val Loss: 0.2731 | Val Acc: 0.9125
Precision: 0.9178 | Recall: 0.9125 | F1: 0.9117




#### Training 2 (Fine-Tuning)

In [13]:
for param in model.parameters():
    param.requires_grad = True


In [14]:
optimizer = optim.AdamW(model.parameters(), lr=1e-5)

In [ ]:
EPOCHS_STAGE_FINETUNE = 20

best_val_acc = 0.0

print("\n-----------Starting Fine-tuning Stage-----------\n")

for epoch in range(EPOCHS_STAGE_FINETUNE):

    train_loss, train_acc = train_one_epoch(train_loader, epoch, EPOCHS_STAGE_FINETUNE)
    precision, recall, f1, val_loss, val_acc = validate(val_loader)

    print(f"\nEpoch [{epoch+1}/{EPOCHS_STAGE_FINETUNE}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")

        
torch.save({
            "model_state": model.state_dict(),
            "class_names": class_names
        }, os.path.join(WEIGHTS_DIR, "ConvNeXt-S.pth"))


-----------Starting Fine-tuning Stage-----------



Validating: 100%|██████████| 8/8 [00:49<00:00,  6.16s/it]



Epoch [1/20]
Train Loss: 0.1925 | Train Acc: 0.9406
Val Loss: 0.2594 | Val Acc: 0.9167
Precision: 0.9206 | Recall: 0.9167 | F1: 0.9164


Validating: 100%|██████████| 8/8 [00:48<00:00,  6.08s/it]



Epoch [2/20]
Train Loss: 0.1360 | Train Acc: 0.9552
Val Loss: 0.2093 | Val Acc: 0.9292
Precision: 0.9319 | Recall: 0.9292 | F1: 0.9292


...


Epoch [18/20]
Train Loss: 0.0120 | Train Acc: 0.9990
Val Loss: 0.1099 | Val Acc: 0.9708
Precision: 0.9726 | Recall: 0.9708 | F1: 0.9711


Validating: 100%|██████████| 8/8 [00:50<00:00,  6.37s/it]



Epoch [19/20]
Train Loss: 0.0110 | Train Acc: 0.9990
Val Loss: 0.1121 | Val Acc: 0.9750
Precision: 0.9773 | Recall: 0.9750 | F1: 0.9753


Validating: 100%|██████████| 8/8 [00:51<00:00,  6.38s/it]


Epoch [20/20]
Train Loss: 0.0093 | Train Acc: 0.9990
Val Loss: 0.1140 | Val Acc: 0.9750
Precision: 0.9773 | Recall: 0.9750 | F1: 0.9753


#### Prediction Sample

In [20]:
from PIL import Image

In [21]:
checkpoint = torch.load("../weights/ConvNeXt-S.pth")

class_names = checkpoint["class_names"]

model.load_state_dict(checkpoint["model_state"])
model.eval()

def predict(image_path):

    image = Image.open(image_path).convert("RGB")
    image = val_transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs, 1)

    return class_names[pred.item()]


In [22]:
# Example usage:
prediction = predict(r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\val\Healthy\Healthy_val_13.jpg")
prediction1 = predict(r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\val\Leaf Blight\leaf blight_val_15.jpg")
prediction2 = predict(r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\val\Leaf Spot\leaf spot_val_14.jpg")
prediction3 = predict(r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\val\Spider Mites\Spider Mites_val_24.jpg")
print(f"Healthy Predicted class: {prediction}")
print(f"Leaf Blight Predicted class: {prediction1}")
print(f"Leaf Spot Predicted class: {prediction2}")
print(f"Spider Mites Predicted class: {prediction3}")

Healthy Predicted class: Healthy
Leaf Blight Predicted class: leaf blight
Leaf Spot Predicted class: leaf spot
Spider Mites Predicted class: Spider Mites
